In [ ]:
from syft_rds.orchestra import setup_rds_stack
from rds_chat_analysis import REPO_ROOT
from rds_chat_analysis import DATA_DIR
import shutil


from rds_chat_analysis import NOTEBOOK_DIR

In [ ]:
key = "wildchat"
stack = setup_rds_stack(
    root_dir=REPO_ROOT / ".rds",
    key=key,
    log_level="DEBUG",
    reset=True,
)

do_client = stack.do_rds_client
ds_client = stack.ds_rds_client

In [ ]:
DATASET_NAME = "Wildchat-postgres"

local_data_dir = DATA_DIR / DATASET_NAME
private_dir = local_data_dir / "private"
mock_dir = local_data_dir / "mock"
markdown_path = local_data_dir / "README.md"

shutil.rmtree(local_data_dir, ignore_errors=True)
private_dir.mkdir(parents=True, exist_ok=True)
mock_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
MOCK_CREDENTIALS = NOTEBOOK_DIR / "v2" / "config_mock.example.toml"
PRIVATE_CREDENTIALS = NOTEBOOK_DIR / "v2" / "config_private.example.toml"

_ = shutil.copy(MOCK_CREDENTIALS, mock_dir / "config.toml")
_ = shutil.copy(PRIVATE_CREDENTIALS, private_dir / "config.toml")

print(f"Mock dir structure: {mock_dir}")
for file in mock_dir.iterdir():
    print(f"└──📄 {file.name}")
print(f"Private dir structure: {private_dir}")
for file in private_dir.iterdir():
    print(f"└──📄 {file.name}")

In [ ]:
description_markdown = """
# Wildchat Postgres Dataset
"""

markdown_path.write_text(description_markdown.strip())

In [ ]:
wildchat_dataset = do_client.dataset.create(
    name=DATASET_NAME,
    path=private_dir,
    mock_path=mock_dir,
    summary="A embedded wildchat dataset in postgres.",
    description_path=markdown_path,
)

In [ ]:
wildchat_dataset = ds_client.dataset.get(name=DATASET_NAME)
wildchat_dataset.describe()

# DS submits Job

In [ ]:
from rds_chat_analysis.client import RDSChatAnalysisClient

ds_client = RDSChatAnalysisClient.from_rdsclient(ds_client)

In [ ]:
job = ds_client.submit_job(
    vector_store_query="Messages about food and drinks",
    aggregation_query="Is this conversation about big brand names like Coca Cola, Nestle, or similar? answer with 'Yes.' or 'No.', followed by a short explanation.",
    aggregation_fn="pairwise_qa",
    max_vector_store_results=100,
    distance_threshold=0.4,
    filters={"role": "user"},
)

In [ ]:
job.describe()

## DO reviews and executes

In [ ]:
do_client = RDSChatAnalysisClient.from_rdsclient(do_client)

pending_jobs = do_client.jobs.get_all(status="pending_code_review")

job_to_execute = pending_jobs[0]

# Review the job configuration
do_client.review_job(job_to_execute)

In [ ]:
do_client.run_private(job_to_execute)

In [ ]:
# TODO client.review_results(job)
do_client.jobs.share_results(job_to_execute)

## DS checks results

In [ ]:
job = ds_client.jobs.get_all()[0]

In [ ]:
ds_client.get_job_result(job, output_filename="output/result.json")